# Gradient Boosting Masterclass

Executed notes for functional-gradient theory, scratch regression/classification, scikit-learn models, diagnostics, threshold policy, and production gates.

In [1]:
from pathlib import Path
import sys,numpy as np,pandas as pd
from IPython.display import HTML,display
from sklearn.datasets import load_diabetes,load_breast_cancer
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.metrics import mean_squared_error,roc_auc_score,log_loss,brier_score_loss
ROOT=Path.cwd();sys.path.insert(0,str(ROOT));SEED=42
from src.gradient_boosting_from_scratch import GradientBoostingRegressorScratch,GradientBoostingBinaryClassifierScratch
from src.library_pipeline import build_regressor,build_classifier
def fig(n,c):display(HTML(f'<img src="../figures/{n}" style="max-width:100%"><b>{c}</b>'))
print('ready',SEED)

ready 42


## 1. Additive stage-wise model

$F_M(x)=F_0(x)+\sum_{m=1}^M\nu\rho_mh_m(x)$. Bagging fits independent learners to reduce variance; boosting fits sequential corrections. Learning rate controls step size, depth controls interaction order, leaf size smooths corrections, and subsampling adds stochastic regularization.

In [2]:
pd.DataFrame({'method':['bagging','boosting'],'dependence':['independent','sequential'],'primary effect':['variance reduction','gradient-based error correction'],'parallelism':['high','limited']})

     method   dependence                   primary effect parallelism
0   bagging  independent               variance reduction        high
1  boosting   sequential  gradient-based error correction     limited

## 2. Functional gradient descent and manual rounds

For risk $\sum_iL(y_i,F(x_i))$, pseudo-residual $r_{im}=-\partial L/\partial F(x_i)$. Squared error gives $r=y-F(x)$. Binary log-loss with $p=\sigma(F)$ gives $r=y-p$; Newton leaf correction is $\sum(y-p)/\sum p(1-p)$.

In [3]:
x=np.arange(1,7).reshape(-1,1);y=np.array([2,2.5,3.5,5,7.5,9.])
toy=GradientBoostingRegressorScratch(3,.5,1,1,1.,SEED).fit(x,y)
print('F0',round(toy.init_,3),'stage MSE',[round(v,3) for v in toy.train_loss_]);fig('figure_01.svg','Manual residual correction')

F0 4.917 stage MSE [2.451, 0.946, 0.472]


## 3. Deterministic data and leakage control

The dataset contract is recorded in `data/dataset_manifest.json`. Development data are split into train/validation; test labels remain untouched until final evaluation. Hyperparameters and decision thresholds are selected without test feedback.

In [4]:
r=load_diabetes(as_frame=True);c=load_breast_cancer(as_frame=True)
Xrd,Xrt,yrd,yrt=train_test_split(r.data,r.target,test_size=.25,random_state=SEED);Xr,Xrv,yr,yrv=train_test_split(Xrd,yrd,test_size=.25,random_state=SEED)
Xcd,Xct,ycd,yct=train_test_split(c.data,c.target,test_size=.25,stratify=c.target,random_state=SEED);Xc,Xcv,yc,ycv=train_test_split(Xcd,ycd,test_size=.25,stratify=ycd,random_state=SEED)
print(Xr.shape,Xrv.shape,Xrt.shape,Xc.shape,Xcv.shape,Xct.shape);fig('figure_02.svg','Target structure and split context')

(248, 10) (83, 10) (111, 10) (319, 30) (107, 30) (143, 30)


## 4. Regression from scratch and library implementation

The scratch regressor initializes at the mean, fits shallow trees to residuals, and exposes staged predictions. The library model adds Huber loss, shrinkage, minimum leaf size, stochastic subsampling, validation fraction, and early stopping.

In [5]:
sr=GradientBoostingRegressorScratch(100,.05,2,8,1.,SEED).fit(Xr,yr);fr=build_regressor(SEED).fit(Xrd,yrd);rp=fr.predict(Xrt)
print({'scratch_val_RMSE':round(mean_squared_error(yrv,sr.predict(Xrv))**.5,3),'test_RMSE':round(mean_squared_error(yrt,rp)**.5,3),'test_R2':round(fr.score(Xrt,yrt),3)});fig('figure_03.svg','Regression optimization, baselines, residuals, importance, and partial dependence')

{'scratch_val_RMSE': 63.337, 'test_RMSE': 53.824, 'test_R2': 0.476}


## 5. Regression regularization and diagnostics

Treat learning rate and stage count jointly. Validate depth, leaf size, subsampling, and early stopping. Final review includes residual structure, extreme errors, stage-wise loss, permutation importance, and partial dependence. Predictive importance is not causal.

In [6]:
from sklearn.ensemble import GradientBoostingRegressor
trade={}
for lr,n in [(.1,100),(.05,200),(.03,300)]:
 m=GradientBoostingRegressor(n_estimators=n,learning_rate=lr,max_depth=2,min_samples_leaf=8,subsample=.85,random_state=SEED).fit(Xr,yr);trade[f'{lr}/{n}']=round(mean_squared_error(yrv,m.predict(Xrv))**.5,3)
print(trade);fig('figure_03.svg','Shrinkage and validation diagnostics')

{'0.1/100': 63.398, '0.05/200': 63.363, '0.03/300': 62.687}


## 6. Binary classification from scratch

The scratch classifier fits trees to $y-p$ and applies curvature-aware leaf corrections. Evaluate ranking, probability quality, and decision policy separately; accuracy alone cannot expose calibration or asymmetric costs.

In [7]:
sc=GradientBoostingBinaryClassifierScratch(100,.05,1,8,1.,SEED).fit(Xc,yc);sp=sc.predict_proba(Xcv)[:,1]
print({'scratch_val_AUC':round(roc_auc_score(ycv,sp),4),'scratch_logloss':round(log_loss(ycv,sp),4)});fig('figure_04.svg','Binary log-loss optimization and validation')

{'scratch_val_AUC': 0.9918, 'scratch_logloss': 0.1376}


## 7. Library classifier, calibration, and threshold cost

The probability model and decision threshold are separate governed assets. Choose a threshold on validation data using explicit false-negative/false-positive costs; freeze it before test evaluation and revalidate it when prevalence, costs, or calibration drift.

In [8]:
fc=build_classifier(SEED).fit(Xcd,ycd);p=fc.predict_proba(Xct)[:,1];ts=np.linspace(.05,.95,91);cost=[5*np.sum((yct==1)&(p<t))+np.sum((yct==0)&(p>=t)) for t in ts];bt=float(ts[np.argmin(cost)])
print({'AUC':round(roc_auc_score(yct,p),4),'logloss':round(log_loss(yct,p),4),'Brier':round(brier_score_loss(yct,p),4),'cost_threshold':round(bt,2)});fig('figure_04.svg','ROC/PR, probability quality, threshold cost, and explanation')

{'AUC': 0.9939, 'logloss': 0.0978, 'Brier': 0.031, 'cost_threshold': 0.39}


## 8. Cross-validation, limitations, and production gates

Classical boosting is sequential, hyperparameter-sensitive, vulnerable to noisy labels, and weak at extrapolation. Monitor schema violations, feature/target drift, residual or calibration drift, threshold cost, latency, and rollback conditions.

In [9]:
rcv=-cross_val_score(build_regressor(SEED),Xrd,yrd,cv=3,scoring='neg_root_mean_squared_error');ccv=cross_val_score(build_classifier(SEED),Xcd,ycd,cv=3,scoring='roc_auc')
print('CV RMSE',round(rcv.mean(),3),'+/-',round(rcv.std(),3),'| CV AUC',round(ccv.mean(),4),'+/-',round(ccv.std(),4));fig('figure_03.svg','Regression evidence');fig('figure_04.svg','Classification evidence')

CV RMSE 57.73 +/- 2.062 | CV AUC 0.984 +/- 0.0172


## 9. Reproducibility and artifacts

Use `environment.yaml`, execute from the package root, and run `python scripts/verify_package.py`. The repository commits the executed notebook, static HTML, theory notes, scratch/library source, dataset manifest, metrics, exercises, figures, and validation report.

In [10]:
import json
print(json.loads((ROOT/'artifacts/validation_report.json').read_text()));print(pd.read_csv(ROOT/'artifacts/metrics_summary.csv'))

{'total_cells': 21, 'markdown_cells': 11, 'code_cells': 10, 'unexecuted_code_cells': [], 'error_outputs': [], 'rendered_figure_outputs': 8, 'missing_required_files': [], 'html_bytes': 3649, 'passes': True}
regression test: MAE=41.746 RMSE=52.012 R2=0.489
classification test: ROC_AUC=0.992 AP=0.995 LogLoss=0.110 Brier=0.036 BalancedAcc=0.952 F1=0.973 threshold=0.05


## Final mental model

Gradient boosting is **functional gradient descent constrained to a weak-learner function class**. Reliable deployment requires leakage-safe validation, simple baselines, regularized stage-wise optimization, untouched-test evaluation, residual/calibration/threshold diagnostics, validation-grounded explanations, monitoring, and rollback gates.